# Day 9: Darcy's law in one dimension and the finite difference

**Time:** about 1 hour  |  **Needs:** Days 1 to 8
**Reservoir idea:** how fast a fluid flows through rock for a given pressure difference (Darcy's law, with the oilfield-unit factor 0.001127), and how a pressure *slope* is calculated from pressures at a few blocks
**Python you will learn:** `np.diff`, `np.concatenate`, vectorised calculations on neighbouring items (a *stencil*), and checking a result by hand

By the end of today you will be able to:

1. explain a derivative as a slope, and approximate it with a finite difference,
2. calculate the flow rate through a block of rock with Darcy's law in oilfield units,
3. use `np.diff` to get slopes and block sizes from arrays,
4. calculate the flow between **every pair of neighbouring blocks** of a 1-D reservoir without a loop,
5. check that what flows out of one block flows into the next (conservation), including at a closed boundary.

## 0. Setup and recap

**Recap of Day 8.** `a[1:] - a[:-1]` is the difference between each item and the next; masks select items by a condition; `np.where` is an `if` for a whole array; `np.clip` limits values. Today the shifted-slice trick gets its proper name.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

DARCY_UNITS = 0.001127      # Darcy's law in oilfield units (see Section 3)
print("NumPy", np.__version__, "| ready")

## 1. Story: the factor that was 887 times too big

*(An invented but typical situation.)* A student builds a spreadsheet to estimate how much oil a well can push through the rock. She uses Darcy's law, types permeability in millidarcies, pressure in psi, viscosity in centipoise, and gets an answer of 22 000 barrels per day for a small sandstone block. Her supervisor looks at it for three seconds and says "that is the whole field's production, not one block's". The unit conversion factor, 0.001127, was missing, and $1/0.001127 \approx 887$.

Darcy's law is the single most important equation in reservoir simulation; every flow term in `kusim` is a copy of it. Today we use it for one block, then for a whole row of blocks. And we learn to calculate "differences between neighbours", which is how a computer sees a pressure gradient.

## 2. Math refresher: a derivative is a slope

You know that the derivative $dP/dx$ is the slope of the curve $P(x)$. If a pressure falls from 3000 psi at $x=0$ ft to 2900 psi at $x=1000$ ft, the average slope is

$$\frac{dP}{dx} \approx \frac{P_2-P_1}{x_2-x_1} = \frac{2900-3000}{1000-0} = -0.1 \ \text{psi/ft}$$

This is a **finite difference**: instead of an infinitely small step, we use two points a finite distance apart. A simulator only knows the pressure in a few places (the centre of each block), so every gradient in it is a finite difference.

The slope is **negative** because the pressure falls as $x$ grows. Fluid flows *down* the pressure slope, from high to low pressure; this is why Darcy's law will have a minus sign.

**How good is a finite difference?** Take a pressure profile that is *curved*: $P(x)=3000-0.0002\,x^2$ psi (so the exact slope is $dP/dx=-0.0004\,x$). The finite difference between two points is the slope **at the midpoint** between them. For a parabola this is not just good, it is *exact*. Check it:

In [ ]:
x = np.linspace(0, 1000, 6)                  # 0, 200, ..., 1000 ft
p = 3000 - 0.0002 * x**2
print(p)

slope = (p[1:] - p[:-1]) / (x[1:] - x[:-1])  # finite differences, using Day 8 slices
x_mid = (x[1:] + x[:-1]) / 2                 # the midpoints
print(slope)
print(-0.0004 * x_mid)                        # the exact slope at the midpoints

The two printed arrays are identical. Remember this picture: **a difference of two neighbours is the slope in the middle between them**. Its midpoint, not the first point and not the second, is where the slope belongs.

**Your turn 2.1.** For the pressure profile of a *different* parabola, $P(x)=2800-0.0005\,x^2$ at the points `xs = np.linspace(0, 800, 5)`, calculate the finite-difference slopes `fd_slope` using slices (as above), and the exact slopes at the midpoints `exact_slope` from $dP/dx = -0.001\,x$.

In [ ]:
xs = np.linspace(0, 800, 5)
ps = 2800 - 0.0005 * xs**2

# YOUR CODE HERE
fd_slope = None
exact_slope = None

In [ ]:
# Check (do not edit)
assert len(fd_slope) == 4 and len(exact_slope) == 4, "five points have four gaps between them"
assert np.allclose(fd_slope, [-0.1, -0.3, -0.5, -0.7]), f"fd_slope is {fd_slope}"
assert np.allclose(fd_slope, exact_slope), "for a parabola the two should agree exactly"
print("Correct: finite difference = exact slope at the midpoint")

<details><summary>Solution</summary>

```python
fd_slope = (ps[1:] - ps[:-1]) / (xs[1:] - xs[:-1])
xm = (xs[1:] + xs[:-1]) / 2
exact_slope = -0.001 * xm
```

</details>

## 3. Darcy's law in one dimension

For a fluid flowing through a block of rock in one direction:

$$q = -\,0.001127\;\frac{k\,A}{\mu}\;\frac{dP}{dx} \tag{1}$$

| Symbol | Meaning | Unit |
|---|---|---|
| $q$ | flow rate (reservoir conditions) | bbl/day |
| $k$ | permeability | md |
| $A$ | area across the flow (width $\times$ thickness) | ft² |
| $\mu$ | viscosity | cp |
| $dP/dx$ | pressure slope | psi/ft |
| 0.001127 | converts these units to bbl/day | — |

Read it in words: more permeable rock, a bigger area, or a steeper pressure drop gives **more** flow; a thicker (more viscous) fluid gives **less**. The minus sign makes $q$ positive when the pressure falls in the direction of $x$.

For a straight pressure drop over a length $L$, the slope is $(P_2-P_1)/L$, so with the pressure on the **left** $P_L$ and on the **right** $P_R$:

$$q = 0.001127\;\frac{k\,A}{\mu}\;\frac{P_L-P_R}{L} \tag{2}$$

**Worked example.** $k=150$ md, a block 100 ft wide and 30 ft thick ($A=3000$ ft²), oil of $\mu=2$ cp, $P_L-P_R=100$ psi over $L=1000$ ft:

$$q = 0.001127\times\frac{150\times3000}{2}\times\frac{100}{1000} = 25.36\ \text{bbl/day}$$

Do the arithmetic on paper once, then let Python confirm it.

**Your turn 3.1.** Write `darcy_rate(k, area, mu, p_left, p_right, length)` using equation (2). It must work for single numbers **and** for arrays. Then store in `q_example` the worked example above.

In [ ]:
# YOUR CODE HERE
def darcy_rate(k, area, mu, p_left, p_right, length):
    """Flow rate (bbl/day) from the left to the right, from Darcy's law in oilfield units."""
    pass

q_example = None

In [ ]:
# Check (do not edit)
q_example = darcy_rate(150, 3000, 2.0, 3000, 2900, 1000)
assert abs(q_example - 25.3575) < 1e-3, f"the worked example gives 25.36 bbl/day, got {q_example}"
assert darcy_rate(150, 3000, 2.0, 2900, 3000, 1000) < 0, "if the right side has the higher pressure, the flow is to the left (negative)"
assert abs(darcy_rate(300, 3000, 2.0, 3000, 2900, 1000) - 2 * q_example) < 1e-9, "twice the permeability, twice the rate"
print(f"Correct: {q_example:.2f} bbl/day")

<details><summary>Solution</summary>

```python
def darcy_rate(k, area, mu, p_left, p_right, length):
    """Flow rate (bbl/day) from the left to the right, from Darcy's law in oilfield units."""
    return DARCY_UNITS * k * area / mu * (p_left - p_right) / length

q_example = darcy_rate(150, 3000, 2.0, 3000, 2900, 1000)
```

</details>

### Rate against pressure drop

Equation (2) is a **straight line**: double the pressure difference and you double the rate. With an array of pressure drops, one call gives all the rates:

In [ ]:
dp = np.array([0, 25, 50, 100, 200])                       # psi
q = darcy_rate(150, 3000, 2.0, 3000, 3000 - dp, 1000)
print(q)

plt.figure(figsize=(5, 3.5))
plt.plot(dp, q, "o-")
plt.title("Darcy's law is a straight line")
plt.xlabel("pressure drop $P_L - P_R$ (psi)")
plt.ylabel("flow rate $q$ (bbl/day)")
plt.grid(True)
plt.show()

### From reservoir barrels to stock-tank barrels

Equation (1) gives the flow **in the reservoir**, in *reservoir* barrels per day. A well report is in *stock-tank* barrels (STB), at surface conditions. The oil shrinks on its way up as gas comes out, and the formation volume factor $B_o$ (rb/STB, Day 5) is the ratio:

$$q_{\text{STB}} = \frac{q_{\text{reservoir}}}{B_o}$$

**Your turn 3.2.** The oil in the worked example has $B_o=1.5$ rb/STB. Store in `q_stb` the rate in STB/day, and in `shrinkage_pct` how many percent smaller the surface volume is than the reservoir volume ($100\,(1 - 1/B_o)$).

In [ ]:
# YOUR CODE HERE
q_stb = None
shrinkage_pct = None

In [ ]:
# Check (do not edit)
assert abs(q_stb - 16.905) < 1e-3, f"25.3575 / 1.5 = 16.905, got {q_stb}"
assert abs(shrinkage_pct - 33.333) < 1e-2, f"1 - 1/1.5 is one third, got {shrinkage_pct}"
print(f"Correct: {q_stb:.2f} STB/day; the oil shrinks by {shrinkage_pct:.0f} %")

<details><summary>Solution</summary>

```python
bo = 1.5
q_stb = q_example / bo
shrinkage_pct = 100 * (1 - 1 / bo)
```

</details>

## 4. `np.diff`

You have calculated $a[1:]-a[:-1]$ several times. NumPy has a name for it: **`np.diff(a)`**. It returns the differences between neighbours, so for `n` items it gives `n - 1` values.

**Predict** what `np.diff` returns for each of the three arrays, then run.

In [ ]:
print(np.diff(np.array([5, 8, 6, 6, 10])))
print(np.diff(np.array([3000.0, 2950.0, 2900.0, 2850.0])))
print(np.diff(np.cumsum([1, 2, 3, 4])))

The first gives `[ 3 -2  0  4]` (each item minus the one before it; a fall is negative). The second is `[-50. -50. -50.]`, a steady pressure fall. The third shows that `np.diff` **undoes** `np.cumsum`: the running total of `1 2 3 4` is `1 3 6 10`, and differencing gives back `2 3 4`. (The first value, 1, is lost: the difference of `n` items has `n - 1`.)

Now the pressure slope of a row of blocks. If `p` holds the pressure of each block and `dx` the distance between block centres, the slope between each pair is simply `np.diff(p) / dx`.

**Your turn 4.1.** Six blocks, 200 ft apart (centre to centre), have the pressures `p6` below. Store `dpdx` (the pressure slope between each pair of neighbouring blocks, in psi/ft) and `q_slope_rate` (the flow rate between each pair of neighbours through $k=100$ md, $A=2000$ ft², $\mu=1$ cp, using equation (1): `-DARCY_UNITS * k * area / mu * dpdx`).

In [ ]:
p6 = np.array([3000.0, 2970.0, 2930.0, 2880.0, 2820.0, 2750.0])
dx6 = 200.0

# YOUR CODE HERE
dpdx = None
q_slope_rate = None

In [ ]:
# Check (do not edit)
assert len(dpdx) == 5, "six blocks have five gaps"
assert np.allclose(dpdx, [-0.15, -0.20, -0.25, -0.30, -0.35]), f"dpdx is {dpdx}"
assert np.allclose(q_slope_rate, [33.81, 45.08, 56.35, 67.62, 78.89], atol=1e-2), f"rates are {q_slope_rate}"
print("Correct: the pressure falls ever more steeply, so the flow grows:", np.round(q_slope_rate, 1))

<details><summary>Solution</summary>

```python
dpdx = np.diff(p6) / dx6
q_slope_rate = -DARCY_UNITS * 100 * 2000 / 1.0 * dpdx
```
The flow between the blocks **increases** along the row. That cannot be steady (the same fluid cannot speed up with nothing feeding it): each block receives less than it sends on, so its pressure will change with time. A steady state has the *same* rate through every interface. We come back to this.
</details>

### Block sizes and centres from edges

`kusim` describes a grid by the sizes of its blocks (`dx`), and works out where the **edges** and **centres** are. The edges are the running total of the sizes (with a 0 added at the start), and `np.diff` gives the sizes back. To put the `0.0` at the front of an array we use `np.concatenate`, which joins arrays end to end:

In [ ]:
sizes = np.array([100.0, 100.0, 150.0, 150.0])
edges = np.concatenate(([0.0], np.cumsum(sizes)))
print(edges)
print(np.diff(edges))        # the sizes again

**Your turn 4.2.** A 1-D reservoir has five blocks with the edges `edges5` below. Store `sizes5` (the block sizes), `centres5` (the position of each block centre, halfway between its two edges), and `spacing5` (the distance between the centres of neighbouring blocks).

In [ ]:
edges5 = np.array([0.0, 100.0, 300.0, 600.0, 800.0, 900.0])

# YOUR CODE HERE
sizes5 = None
centres5 = None
spacing5 = None

In [ ]:
# Check (do not edit)
assert np.allclose(sizes5, [100, 200, 300, 200, 100]), f"sizes5 is {sizes5}"
assert np.allclose(centres5, [50, 200, 450, 700, 850]), f"centres5 is {centres5}"
assert np.allclose(spacing5, [150, 250, 250, 150]), f"spacing5 is {spacing5}"
print("Correct")

<details><summary>Solution</summary>

```python
sizes5 = np.diff(edges5)
centres5 = (edges5[1:] + edges5[:-1]) / 2
spacing5 = np.diff(centres5)
```
The spacing between centres is *half the size of one block plus half the size of its neighbour*: for the first pair $(100+200)/2=150$. This is why `kusim` uses both block sizes when it calculates the flow between two blocks.
</details>

## 5. Flow between neighbouring blocks

A reservoir simulator divides the reservoir into blocks, gives each block one pressure (at its centre), and uses Darcy's law **between each pair of neighbours**. Today, in one dimension, with equal blocks of size $\Delta x$, cross-section $A$, permeability $k_i$ and pressure $p_i$ in block $i$.

**The permeability at the face between two blocks.** If the two blocks have different permeabilities, the flow has to pass through half of each. The right average is the **harmonic average**, not the arithmetic one:

$$k_{i+1/2}=\frac{2\,k_i\,k_{i+1}}{k_i+k_{i+1}}$$

(Think of two resistors in series: the weaker rock controls the flow. A 1000 md block next to a 10 md block gives about 20 md, not 505.)

**The flow from block $i$ to block $i+1$** is equation (2) with the centre-to-centre distance $\Delta x$:

$$q_{i+1/2} = \underbrace{0.001127\,\frac{A\,k_{i+1/2}}{\mu\,\Delta x}}_{\text{transmissibility }T_{i+1/2}}\,\bigl(p_i-p_{i+1}\bigr) \tag{3}$$

The part in front of the pressure difference depends only on the rock, the fluid and the geometry, and has a name: the **transmissibility** $T$ (bbl/day per psi). The simulator calculates all the $T$ first, and then the flows.

### By hand (15 minutes)

Three blocks, $\Delta x=100$ ft, $A=1000$ ft², $\mu=1$ cp. Fill in the table on paper, then confirm in code.

| Block | $k$ (md) | $p$ (psi) |
|---|---|---|
| 1 | 100 | 3000 |
| 2 | 50 | 2950 |
| 3 | 100 | 2900 |

1. $k_{1+1/2}$ (between blocks 1 and 2) and $k_{2+1/2}$ (between 2 and 3).
2. $T_{1+1/2}$ and $T_{2+1/2}$ from the part in front of the bracket of equation (3).
3. The flows $q_{1+1/2}$ and $q_{2+1/2}$.

(Check your answers against the check cell of Exercise 5.1.)

**Your turn 5.1.** Write `interface_flows(p, k, dx, area, mu)` that returns an array with the flow through **every interface** between neighbouring blocks (one value fewer than the number of blocks), positive when the flow goes from the lower to the higher block number. Use arrays and slices (`k[:-1]`, `k[1:]`, `np.diff`), no loop. Compute (a) the harmonic face permeabilities, (b) the transmissibilities, (c) the flows.

In [ ]:
# YOUR CODE HERE
def interface_flows(p, k, dx, area, mu):
    """Flow (bbl/day) through each interface of a row of equal blocks, left to right."""
    pass

In [ ]:
# Check (do not edit)
p3 = np.array([3000.0, 2950.0, 2900.0])
k3 = np.array([100.0, 50.0, 100.0])
q3 = interface_flows(p3, k3, dx=100.0, area=1000.0, mu=1.0)
assert len(q3) == 2, "three blocks have two interfaces"
assert np.allclose(q3, [37.567, 37.567], atol=1e-3), f"q3 is {q3}"
q_same = interface_flows(np.array([3000.0, 2900.0]), np.array([100.0, 100.0]), 1000.0, 3000.0, 2.0)
assert abs(q_same[0] - 0.001127 * 100 * 3000 / 2 * 100 / 1000) < 1e-9, "with equal permeabilities this is equation (2)"
print("Correct: 37.57 bbl/day through each face (the low-permeability middle block controls the flow)")

<details><summary>Solution</summary>

```python
def interface_flows(p, k, dx, area, mu):
    """Flow (bbl/day) through each interface of a row of equal blocks, left to right."""
    k_face = 2 * k[:-1] * k[1:] / (k[:-1] + k[1:])        # harmonic average
    trans = DARCY_UNITS * area * k_face / (mu * dx)       # transmissibility
    return trans * (p[:-1] - p[1:])                       # = -trans * np.diff(p)
```
The harmonic average of 100 and 50 md is 66.7 md. Because the middle block is a bottleneck, the face permeability (66.7) is closer to the smaller value than the arithmetic mean (75) would be.
</details>

### What goes in, what comes out

Each block receives the flow from the interface on its left and loses the flow through the interface on its right:

$$\text{net flow into block } i = q_{i-1/2} - q_{i+1/2}$$

The two end blocks have only one interface each. At the outer edges nothing crosses: a **no-flow (closed) boundary**. We can express it by adding a **zero** flow at each end of the array of interface flows, with `np.concatenate`. Then every block has a "left" and a "right" interface, and the net flow is the **negative difference** of the padded flows:

```python
q_padded = np.concatenate(([0.0], q, [0.0]))     # zero flow through the two outer edges
net = -np.diff(q_padded)                        # = q_padded[:-1] - q_padded[1:]
```

**Your turn 5.2.** Write `net_flows(q)` that takes the array of interface flows and returns the net flow **into** each block, with closed ends. Then store `net3` for the `q3` of the previous exercise, and `total_net3` (the sum of all the net flows).

In [ ]:
p3 = np.array([3000.0, 2950.0, 2900.0])
k3 = np.array([100.0, 50.0, 100.0])
q3 = interface_flows(p3, k3, dx=100.0, area=1000.0, mu=1.0)

# YOUR CODE HERE
def net_flows(q):
    """Net flow (bbl/day) into each block from the interface flows q, with closed outer ends."""
    pass

net3 = None
total_net3 = None

In [ ]:
# Check (do not edit)
net3 = net_flows(q3)
total_net3 = net3.sum()
assert len(net3) == 3, "three blocks, three net flows"
assert np.allclose(net3, [-37.567, 0.0, 37.567], atol=1e-3), f"net3 is {net3}"
assert abs(total_net3) < 1e-9, "what leaves one block enters the next: the total is zero"
print("Correct: block 1 loses 37.57, block 2 passes everything through, block 3 gains 37.57")

<details><summary>Solution</summary>

```python
def net_flows(q):
    """Net flow (bbl/day) into each block from the interface flows q, with closed outer ends."""
    q_padded = np.concatenate(([0.0], q, [0.0]))
    return -np.diff(q_padded)

net3 = net_flows(q3)
total_net3 = net3.sum()
```
The total is zero because the boundaries are closed: nothing is created, nothing leaves. This is the simplest **material balance** check, and it returns on Day 18. In `kusim`, a closed face is made by giving the block itself as its own neighbour, so the pressure difference is zero; it is the same trick in a different form (see the bridge).
</details>

## 6. Predict, then run: a barrier in the middle

Take a row of six blocks with a straight pressure drop from 3000 to 2900 psi, and make the permeability uneven: a poor block (10 md) and a mediocre one (50 md) in the middle.

| Block | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| $k$ (md) | 100 | 100 | 10 | 50 | 100 | 100 |

**Predict** (write in the cell below, before running):

1. Which of the five interfaces carries the **least** flow?
2. Will the flow be the *same* through every interface (as in a steady state)?
3. Which block will **gain** fluid (net inflow > 0), and which will lose?

In [ ]:
# 1.
# 2.
# 3.

In [ ]:
k6 = np.array([100.0, 100.0, 10.0, 50.0, 100.0, 100.0])
p6 = np.linspace(3000, 2900, 6)

q6 = interface_flows(p6, k6, dx=200.0, area=2000.0, mu=1.0)
net6 = net_flows(q6)
print("interface flows:", np.round(q6, 2))
print("net into blocks:", np.round(net6, 2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(np.arange(1, 6) + 0.5, q6, width=0.8, color="tab:blue")
axes[0].set_title("Flow through each interface")
axes[0].set_xlabel("between block $i$ and $i+1$ (position $i + 0.5$)")
axes[0].set_ylabel("flow (bbl/day)")
axes[1].bar(np.arange(1, 7), net6, color=np.where(net6 >= 0, "tab:green", "tab:red"))
axes[1].set_title("Net flow into each block (green: gains, red: loses)")
axes[1].set_xlabel("block")
axes[1].set_ylabel("net flow (bbl/day)")
for ax in axes:
    ax.grid(True)
plt.tight_layout()
plt.show()

The face permeabilities are 100, 18.2, 16.7, 66.7 and 100 md. Every interface has the same pressure drop (20 psi), so the flows are proportional to them, and the **smallest flow is through the interface between blocks 3 and 4** (the 10 md block next to the 50 md block). The flow is clearly *not* the same through every interface, so the blocks gain and lose fluid: block 2 receives a lot through its 100 md face but passes little through its 18 md face, so it gains; block 3 and the closed end block 6 gain too; blocks 1, 4 and 5 lose. A block that gains fluid will rise in pressure and one that loses will fall, so **the pressures change until every interface carries the same flow**. Finding those pressures is what a simulator does at each time step, and the next days teach you how.

**Your turn 6.1.** From the arrays `q6` and `net6` above, store `weakest` (the **position**, counting from 0, of the interface with the smallest flow, using `argmin`) and `n_gaining` (how many blocks have a net inflow greater than 0, using a mask).

In [ ]:
# YOUR CODE HERE
weakest = None
n_gaining = None

In [ ]:
# Check (do not edit)
assert weakest == 2, f"the weakest interface is the third one (position 2, between blocks 3 and 4), got {weakest}"
assert n_gaining == 3, f"three blocks gain (blocks 2, 3 and 6), got {n_gaining}"
print("Correct: the weakest face is between blocks 3 and 4; three blocks gain fluid")

<details><summary>Solution</summary>

```python
weakest = q6.argmin()
n_gaining = (net6 > 0).sum()
```
Blocks 2 and 3 gain because what they receive through the face on their left is larger than what leaves through the face on their right (100 against 18.2 md, and 18.2 against 16.7 md). Block 6 gains because it is at the closed end: fluid arrives, nothing leaves.
</details>

## 7. Bridge: Darcy's law inside `kusim`

Open `src/kusim/constants.py`. `DARCY = 0.006328` is your `DARCY_UNITS` times 5.615, because `kusim` works in **cubic feet per day per unit volume of rock**, not barrels. (The 5.615 ft³/bbl you used on Day 4.)

Now open `src/kusim/simulator.py` and find `_build_geometric_factors`. Its formula is

```python
harmonic = 1.0 / (size[nb] / perm[nb] + size / perm)
self.geometry[direction] = DARCY * (2.0 / size) * harmonic * g.is_open[direction]
```

Compare with Section 5:

- `harmonic` is the harmonic average in another form: for equal blocks, $1/(\Delta x/k_{i+1}+\Delta x/k_i) = k_{i+1/2}/(2\Delta x)$, so `(2.0 / size) * harmonic` becomes $k_{i+1/2}/\Delta x^2$. This is our $T$ with the area **divided out**: `kusim` divides every flow by the **block volume** ($A\,\Delta x$), which is why the area does not appear.
- `g.is_open` is 0 at a closed edge and 1 elsewhere, so the transmissibility there is zero: a no-flow boundary again.
- The flow into a block is later added up as `flow += t_o[d] * (p_new[g.neighbour[d]] - p_new)`: the **neighbour's pressure minus this block's**, exactly your `-np.diff` idea. At a closed edge, `g.neighbour` is the block itself, so the difference is zero. In our notebook we added a zero flow with `np.concatenate`; `kusim` makes it with the neighbour trick, and the result is the same.

`kusim` keeps one array per direction (`"x+"`, `"x-"`, `"y+"`, ...) because it works in 3-D; today's 1-D row has only two directions, left and right.

## 8. Self-quiz

Write short answers as comments, then open the answers.

1. A pressure of 3000 psi falls to 2950 psi over 500 ft. What is the slope $dP/dx$, and what is its sign telling you about the direction of flow?
2. What does `np.diff(np.array([2, 5, 4, 10]))` return, and why does it have one item fewer?
3. Why is the factor 0.001127 needed? What goes wrong if you forget it?
4. A 1000 md block touches a 10 md block. Why is the permeability at the face about 20 md and not 505?
5. Interface flows are `q = [30, 30, 20, 20]`. What are the net flows into the five blocks, with closed ends? Do they add up to zero?

In [ ]:
# 1.
# 2.
# 3.
# 4.
# 5.

<details><summary>Answers</summary>

1. $dP/dx=(2950-3000)/500=-0.1$ psi/ft. The negative sign means the pressure falls as $x$ grows, so the fluid flows in the **positive** $x$ direction (down the pressure slope).
2. `[ 3 -1  6]`. Each value is the difference between a neighbour pair, and four items have only three pairs.
3. It converts md, ft², cp, psi and ft into bbl/day. Without it the rate is about 887 times too large.
4. The flow must pass through both rocks in series, so the poor rock controls it, like two resistors in series. The harmonic average $2\cdot1000\cdot10/(1000+10)=19.8$ md reflects this; the arithmetic mean would hide the barrier.
5. Pad with zeros: `[0, 30, 30, 20, 20, 0]`. The net flow is the negative difference, `[-30, 0, 10, 0, 20]`: block 1 loses 30; block 2 gets 30 and loses 30; block 3 gets 30 and loses 20 (+10); block 4 gets 20 and loses 20; block 5 gets 20 and has a closed end (+20). The total is $-30+0+10+0+20=0$.
</details>

## 9. Pocket reference (today's Python and formulas)

| You want to... | Write |
|---|---|
| differences between neighbours | `np.diff(a)` (one item fewer) |
| a pressure slope | `np.diff(p) / dx` |
| block sizes from edges | `np.diff(edges)` |
| edges from sizes | `np.concatenate(([0.0], np.cumsum(sizes)))` |
| centres from edges | `(edges[1:] + edges[:-1]) / 2` |
| join arrays | `np.concatenate((a, b, c))` |
| pad a flow array with zeros | `np.concatenate(([0.0], q, [0.0]))` |
| net flow into each block | `-np.diff(q_padded)` |
| harmonic average of two permeabilities | `2 * k1 * k2 / (k1 + k2)` |
| Darcy flow (bbl/day) | `0.001127 * k * A / mu * (p_left - p_right) / L` |
| reservoir to surface barrels | `q_res / Bo` |

## 10. Wrap-up

Today you:

- saw the derivative as a slope, and the finite difference as the slope at a midpoint,
- used Darcy's law in oilfield units and checked the sizes of the numbers,
- learned `np.diff` and `np.concatenate`,
- calculated the flow through every interface of a row of blocks, with harmonic permeabilities and without loops,
- found the net flow into each block and checked that the total is zero with closed ends,
- saw that a straight pressure drop through uneven rock is *not* a steady state,
- read the same ideas in `kusim`'s constants and in `_build_geometric_factors`.

**Next, Day 10:** gridding: how blocks are numbered, how to find the neighbours of a block (and what happens at an edge), 2-D and 3-D arrays, `reshape`, and turning a flat block number into `(i, j, k)`. It is also the **end of Week 2**, with a consolidation exercise: build a `Grid` of your own and compare it with `grid.py`.